In [8]:
import equinox as eqx
import numpy as np
import jax
import jax.numpy as jnp
import jax.nn as jnn
import jax.random as jran
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
print(f"Pytorch version: {torch.__version__}")
from math import prod
print(f"JAX version: {jax.__version__}")
import optax
print(f"Optax version: {optax.__version__}")
import jax.tree_util as jtu
# import jax.numpy.linalg as jla
# from jax import jit, vmap
from jaxtyping import Array, Float, Int, PyTree # https://github.com/google/jaxtyping
from typing import Callable
from tqdm import tqdm

Pytorch version: 2.12.0
JAX version: 0.10.0
Optax version: 0.2.8


In [ ]:
class NN_Build(eqx.Module):
    layers: list
    input_shape: tuple = eqx.field(static=True)
    output_dim: int = eqx.field(static=True)
    layer_dims: tuple = eqx.field(static=True)
    activation: Callable = eqx.field(static=True)
    key: jax.Array = eqx.field(static=True)

    def __init__(self, 
                 input_shape: tuple,
                 output_dim: int,
                 layer_dims: list,
                 activation: jax.nn = jnn.relu,
                 key: jax.random.PRNGKey=jax.random.PRNGKey(42),
                 ) :
        # PyTree structure for the layers of the neural network. The layers will be stored in a list, and each layer will be added according to the dimensions specified in layer_dims.
        # NamedTuple structure for the layers of the neural network.
        self.layers = []

        # layer_dimss is a list of tuples, where each tuple represents the size of the layer. 
        # For a linear layer, the tuple will have two elements (input_dim, output_dim). 
        # For a convolutional layer, the tuple will have three elements (input_channels, output_channels, kernel_size).
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        if len(self.layer_dims) == 0:
            raise ValueError("At least one layer dimension must be specified in layer_dims.")
        self.activation = activation
        self.key = jax.random.PRNGKey(0) if key is None else key
        self.input_shape = input_shape
        self.output_dim = output_dim

        # Check if the layer dimensions are compatible with the input shape and output dimension. This function will raise an error if the dimensions do not match.
        self.check_layer_dims(self.input_shape, self.output_dim, self.layer_dims)
        print("Layer dimensions are compatible with input shape and output dimension.")

        # Model building loop according to the layer dimensions provided. The first layer is added separately to handle the input shape, and then the rest of the layers are added in a loop.
        keys = jax.random.split(self.key, len(self.layer_dims))
        if len(self.layer_dims) == 1:
            self._add_layer(self.layer_dims[0], None, self.layers, final_layer=True, key=keys[0])
        else:
            self._add_layer(self.layer_dims[0], None, self.layers, final_layer=False, key=keys[0])
            size_prev = self.layer_dims[0]
            for i, size_new in enumerate(self.layer_dims[1:-1]):
                self._add_layer(size_new, size_prev, self.layers, final_layer=False, key=keys[i+1])
                size_prev = size_new
            self._add_layer(self.layer_dims[-1], size_prev, self.layers, final_layer=True, key=keys[-1])
            size_prev = self.layer_dims[-1]


    def check_layer_dims(self,
                         input_shape: tuple,
                         output_dim: int,
                         layer_dims: list):
        # The function checks if the final data shape after passing through all layers matches the specified output dimension.
        # data_shape is the actual shape of the data as it passes through the layers of the neural network. 
        data_shape = input_shape
        for layer_dim in layer_dims:
            if len(layer_dim) not in [2, 3]:
                raise ValueError(f"Each layer dimension must be a tuple of length 2 (for linear layers) or 3 (for convolutional layers). Got {layer_dim}.")
            elif len(layer_dim) == 2:
                flattened_shape = prod(data_shape)
                if flattened_shape != layer_dim[0]:
                    raise ValueError(f"Expected input shape {data_shape} to be compatible with linear layer input dimension {layer_dim[0]}.")
                data_shape = (layer_dim[1],)
            elif len(layer_dim) == 3:
                if len(data_shape) == 3:
                    if data_shape[2] != layer_dim[0]:
                        raise ValueError(f"Expected input shape {data_shape} to be compatible with convolutional layer input channels {layer_dim[0]}.")
                    new_h = data_shape[0] - (layer_dim[2] - 1)
                    new_w = data_shape[1] - (layer_dim[2] - 1)
                    if new_h <= 0 or new_w <= 0:
                        raise ValueError(f"Kernel size {layer_dim[2]} is too large for input shape {data_shape}.")
                    data_shape = (new_h, new_w, layer_dim[1])
                elif len(data_shape) != 3:
                    raise ValueError(f"Cannot add a convolutional layer after a linear layer, got layer dimensions {layer_dim} after input shape {data_shape}.")
        if data_shape != (output_dim,):
            raise ValueError(f"Final layer shape {data_shape} does not match output dimension {output_dim}.")

    def _add_layer(self, 
                   layer_dim: tuple, 
                   prev_layer_dim: tuple, 
                   layers: list,
                   final_layer: bool = False,
                   key: jax.random.PRNGKey = None):
        # Special case for the first layer with no previous layer dimensions.
        if prev_layer_dim is None:
            if len(layer_dim) == 2:
                if len(self.input_shape) == 1:
                    if self.input_shape[0] != layer_dim[0]:
                        raise ValueError(
                            f"Expected input shape {self.input_shape} to match linear "
                            f"input dimension {layer_dim[0]}."
                        )
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=key))
                elif len(self.input_shape) == 3:
                    if prod(self.input_shape) != layer_dim[0]:
                        raise ValueError(
                            f"Expected flattened input size {prod(self.input_shape)} to "
                            f"match linear input dimension {layer_dim[0]}."
                        )
                    layers.append(jnp.ravel)
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=key))
            elif len(layer_dim) == 3: 
                if len(self.input_shape) != 3:
                    raise ValueError(f"Expected input shape {self.input_shape} to be compatible with convolutional layer input shape, got {layer_dim} after input shape {self.input_shape}.")
                if self.input_shape[2] != layer_dim[0]:
                    raise ValueError(f"Expected input shape {self.input_shape} to be compatible with convolutional layer input channels {layer_dim[0]}.")   
                elif len(self.input_shape) == 3:     
                    layers.append(eqx.nn.Conv2d(layer_dim[0], layer_dim[1], kernel_size=layer_dim[2], key=key))
                elif len(self.input_shape) != 3:
                    raise ValueError(f"Cannot add a convolutional layer after a linear layer, got layer dimensions {layer_dim} after input shape {self.input_shape}.")
        # Adding rest of the layers according to the dimensions specified in layer_dim and prev_layer_dim. 
        # The function checks the dimensions and adds the appropriate layer (linear or convolutional) followed by the activation function.
        # No padding, stride of 1, and no dilation are assumed for convolutional layers.
        else:
            if len(layer_dim) == 2: 
                if len(prev_layer_dim) == 2:
                    if prev_layer_dim[1] != layer_dim[0]:
                        raise ValueError(f"Expected previous layer output dimension {prev_layer_dim[1]} to match linear input dimension {layer_dim[0]}.")
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=key))
                elif len(prev_layer_dim) == 3:
                    layers.append(jnp.ravel)
                    layers.append(eqx.nn.Linear(layer_dim[0], layer_dim[1], key=key))
            elif len(layer_dim) == 3:
                if len(prev_layer_dim) == 2:
                    raise ValueError(f"Cannot add a convolutional layer after a linear layer, got layer dimensions {layer_dim} after {prev_layer_dim}.")
                elif len(prev_layer_dim) == 3:
                    if prev_layer_dim[1] != layer_dim[0]:
                        raise ValueError(f"Expected previous layer output channels {prev_layer_dim[1]} to match linear input dimension {layer_dim[0]}.")
                    layers.append(eqx.nn.Conv2d(layer_dim[0], layer_dim[1], kernel_size=layer_dim[2], key=key))
        # If this is not the final layer, we add the activation function.
        if not final_layer:
            layers.append(self.activation)

    def __call__(self, x):
        for l, layer in enumerate(self.layers):
            x = layer(x)
        return x
    


    

In [24]:
input_shape = (60,60,5)
output_dim = 6
layer_dims = ((5,1,5) ,(56*56, 64), (64, 32), (32, output_dim))
CNN = NN_Build(input_shape, output_dim, layer_dims)
print(CNN)

@eqx.filter_jit
def mse_loss(model, x, y):
    preds = jax.vmap(model)(x)
    return jnp.mean((preds - y) ** 2)

with np.load('../sbi_lens_sims/mpi_job_4483820_combined.npz') as data:
    x = data['y']
    # Jax Conv layers expects data shape in (N, C, H, W) format, but the data is in (N, H, W, C) format. 
    # We need to transpose the data to match the expected input shape for the convolutional layers.
    x = jnp.transpose(x, (0, 3, 1, 2))  # NHWC -> NCHW 
    y = data['theta']
    # Remove any NaN data points
    nan_mask = jnp.isnan(x).reshape(x.shape[0], -1).any(axis=1)
    clean_indices = jnp.where(~nan_mask)[0]
    x = x[clean_indices]
    y = y[clean_indices]
    print("x", x.shape, "\n", "y", y.shape)

# Filter spec approach for partitioning model into params and static:
# 1. jtu.tree_map(lambda _: False, model) sets all non-layer fields to False
# 2. jtu.tree_map(eqx.is_array, model.layers) recursively checks each leaf in layers (arrays→True, modules→False)
# 3. eqx.tree_at replaces only the .layers subtree with the correct spec
# Result: non-layer fields are False, layer arrays are True for proper partitioning
filter_spec = eqx.tree_at(
    lambda m: m.layers,
    jtu.tree_map(lambda _: False, CNN),  # Step 1: ALL fields → False
    jtu.tree_map(eqx.is_array, CNN.layers)  # Step 2: layers → array mask
)

loss = mse_loss(CNN, x, y)
# params, static = eqx.partition(CNN, eqx.is_inexact_array)
params, static = eqx.partition(CNN, filter_spec)
print("params", params)
print("static", static)

Layer dimensions are compatible with input shape and output dimension.
NN_Build(
  layers=[
    Conv2d(
      num_spatial_dims=2,
      weight=f32[1,5,5,5],
      bias=f32[1,1,1],
      in_channels=5,
      out_channels=1,
      kernel_size=(5, 5),
      stride=(1, 1),
      padding=((0, 0), (0, 0)),
      dilation=(1, 1),
      groups=1,
      use_bias=True,
      padding_mode='ZEROS'
    ),
    <PjitFunction of <function relu at 0x113375620>>,
    <PjitFunction of <function ravel at 0x113150180>>,
    Linear(
      weight=f32[64,3136],
      bias=f32[64],
      in_features=3136,
      out_features=64,
      use_bias=True
    ),
    <PjitFunction of <function relu at 0x113375620>>,
    Linear(
      weight=f32[32,64],
      bias=f32[32],
      in_features=64,
      out_features=32,
      use_bias=True
    ),
    <PjitFunction of <function relu at 0x113375620>>,
    Linear(
      weight=f32[6,32],
      bias=f32[6],
      in_features=32,
      out_features=6,
      use_bias=True
    )


/var/folders/yt/ybnb7bw10vj2y7213d6y2l_00000gn/T/ipykernel_58974/2360748947.py:4: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  CNN = NN_Build(input_shape, output_dim, layer_dims)


x (4872, 5, 60, 60) 
 y (4872, 6)
params NN_Build(
  layers=[
    Conv2d(
      num_spatial_dims=2,
      weight=f32[1,5,5,5],
      bias=f32[1,1,1],
      in_channels=5,
      out_channels=1,
      kernel_size=(5, 5),
      stride=(1, 1),
      padding=((0, 0), (0, 0)),
      dilation=(1, 1),
      groups=1,
      use_bias=True,
      padding_mode='ZEROS'
    ),
    None,
    None,
    Linear(
      weight=f32[64,3136],
      bias=f32[64],
      in_features=3136,
      out_features=64,
      use_bias=True
    ),
    None,
    Linear(
      weight=f32[32,64],
      bias=f32[32],
      in_features=64,
      out_features=32,
      use_bias=True
    ),
    None,
    Linear(
      weight=f32[6,32],
      bias=f32[6],
      in_features=32,
      out_features=6,
      use_bias=True
    )
  ],
  input_shape=(60, 60, 5),
  output_dim=6,
  layer_dims=((5, 1, 5), (3136, 64), (64, 32), (32, 6)),
  activation=<PjitFunction of <function relu at 0x113375620>>,
  key=u32[2]
)
static NN_Build(
  layer

In [25]:
def evaluate(model, loss, testloader: torch.utils.data.DataLoader):
    """This function evaluates the model on the test dataset,
    computing both the average loss and the average accuracy.
    """
    avg_loss = 0
    for x, y in testloader:
        x = x.numpy()
        y = y.numpy()
        # Note that all the JAX operations happen inside `loss` and `compute_accuracy`,
        # and both have JIT wrappers, so this is fast.
        avg_loss += loss(model, x, y)
    return avg_loss / len(testloader)

loss, grads = eqx.filter_value_and_grad(mse_loss)(CNN, x, y)
output = jax.vmap(CNN)(x)
print('output', output.shape)

output (4872, 6)


In [26]:
def train(
    model,
    loss: Callable,
    trainloader: torch.utils.data.DataLoader,
    testloader: torch.utils.data.DataLoader,
    optim: optax.GradientTransformation,
    steps: int,
    print_every: int,
) -> CNN:
    # Just like earlier: It only makes sense to train the arrays in our model,
    # so filter out everything else.
    opt_state = optim.init(eqx.filter(model, eqx.is_array))
    filter_spec = eqx.tree_at(
    lambda m: m.layers,
    jtu.tree_map(lambda _: False, model),  # Step 1: ALL fields → False
    jtu.tree_map(eqx.is_array, model.layers)  # Step 2: layers → array mask
    )
    params, static = eqx.partition(model, filter_spec)
    opt_state = optim.init(params)

    # Always wrap everything -- computing gradients, running the optimiser, updating
    # the model -- into a single JIT region. This ensures things run as fast as
    # possible.
    @eqx.filter_jit
    def make_step(
        model,
        opt_state: PyTree,
        x: Float[Array, "batch 1 28 28"],
        y: Int[Array, " batch"],
        ):
        # filter_spec = eqx.tree_at(
        # lambda m: m.layers,
        # jtu.tree_map(lambda _: False, model),  # Step 1: ALL fields → False
        # jtu.tree_map(eqx.is_array, model.layers)  # Step 2: layers → array mask
        # )
        # params, static = eqx.partition(model, filter_spec)
        loss_value, grads = eqx.filter_value_and_grad(loss)(model, x, y)
        updates, opt_state = optim.update(
            grads, opt_state, eqx.filter(model, eqx.is_array)
        )
        model = eqx.apply_updates(model, updates)
        return model, opt_state, loss_value

    @eqx.filter_jit
    def evaluate(model, loss, testloader: torch.utils.data.DataLoader):
        """This function evaluates the model on the test dataset,
        computing both the average loss and the average accuracy.
        """
        avg_loss = 0
        for x, y in testloader:
            x = x.numpy()
            y = y.numpy()
            # Note that all the JAX operations happen inside `loss` and `compute_accuracy`,
            # and both have JIT wrappers, so this is fast.
            avg_loss += loss(model, x, y)
        return avg_loss / len(testloader)

    # Loop over our training dataset as many times as we need.
    def infinite_trainloader():
        while True:
            yield from trainloader

    for step, (x, y) in tqdm(zip(range(steps), infinite_trainloader())):
        # PyTorch dataloaders give PyTorch tensors by default,
        # so convert them to NumPy arrays.
        x = x.numpy()
        y = y.numpy()
        model, opt_state, train_loss = make_step(model, opt_state, x, y)
        if (step % print_every) == 0 or (step == steps - 1):
            test_loss = evaluate(model, loss, testloader)
            print(
                f"{step=}, train_loss={train_loss.item()}, "
                f"test_loss={test_loss.item()}"
            )
    return model

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
STEPS = 15000
PRINT_EVERY = 3000
# SEED = 5678
TRAIN_TEST_SPLIT = 0.8

optim = optax.adamw(LEARNING_RATE)

# Create torch Datasets and corresponding train-test DataLoaders
x_tensor = torch.tensor(np.array(x), dtype=torch.float32)
y_tensor = torch.tensor(np.array(y), dtype=torch.float32)
dataset = TensorDataset(x_tensor, y_tensor)
train_size = int(TRAIN_TEST_SPLIT * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

# Train Model
model = train(CNN, mse_loss, train_loader, test_loader, optim, STEPS, PRINT_EVERY)

Train size: 3897, Test size: 975


11it [00:00, 14.65it/s]

step=0, train_loss=0.618362307548523, test_loss=0.6401203870773315


3012it [00:27, 104.56it/s]

step=3000, train_loss=0.014548778533935547, test_loss=0.0674094706773758


...

step=9000, train_loss=0.005905551370233297, test_loss=0.10348792374134064


12021it [01:48, 106.33it/s]

step=12000, train_loss=0.002321828855201602, test_loss=0.12364249676465988


15000it [02:15, 111.03it/s]

step=14999, train_loss=0.0009812756907194853, test_loss=0.13491196930408478


In [46]:
print(f"X range: {x.min():.2f} to {x.max():.2f}")
print(f"Y range: {y.min():.2f} to {y.max():.2f}")


X range: -0.17 to 0.46
Y range: -2.00 to 1.31


In [59]:
print("Any NaNs in X?", jnp.any(jnp.isnan(x)))
print("Any NaNs in Y?", jnp.any(jnp.isnan(y)))
print("Any Infs in X?", jnp.any(jnp.isinf(x)))
print("Any Infs in Y?", jnp.any(jnp.isinf(y)))
print(jnp.isnan(x).reshape(x.shape[0], -1).any(axis=1))
clean_indices = jnp.where(~nan_mask)[0]
print(clean_indices)

Any NaNs in X? False
Any NaNs in Y? False
Any Infs in X? False
Any Infs in Y? False
[False False False ... False False False]
[   0    1    2 ... 4869 4870 4871]


In [36]:
# x, y = test_loader
for x, y in test_loader:
    print(x.shape,y.shape)
print(type(test_loader))


torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([64, 5, 60, 60]) torch.Size([64, 6])
torch.Size([15, 5, 60, 60]) torch.Size([15, 6])
<class 'torch.utils.data.dataloader.DataLoader'>
